# Feature Engineering:

Features created:
* temporal features
* transaction amount features
* geographical distance features
* missingness indicators
* card/customer history features
* interaction features
* cautious frequency features

Note: to avoid temporal leakage, history and frequency freatures are calculated only on previous transactions to evaluate future data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

In [2]:
DATA_DIR = Path("../data/raw")

SPARKOV_TRAIN_PATH = DATA_DIR / "fraudTrain.csv"
SPARKOV_TEST_PATH = DATA_DIR / "fraudTest.csv" 

IEEE_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IEEE_IDENTITY_PATH = DATA_DIR / "train_identity.csv"

def read_csv_checked(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"File not found: {Path(path).resolve()}")
    print(f"Reading {path} ...")
    return pd.read_csv(path, low_memory=False)

sparkov_train = read_csv_checked(SPARKOV_TRAIN_PATH)
sparkov_test = read_csv_checked(SPARKOV_TEST_PATH)

ieee_transaction = read_csv_checked(IEEE_TRANSACTION_PATH)
ieee_identity = read_csv_checked(IEEE_IDENTITY_PATH)

Reading ..\data\raw\fraudTrain.csv ...
Reading ..\data\raw\fraudTest.csv ...
Reading ..\data\raw\train_transaction.csv ...
Reading ..\data\raw\train_identity.csv ...


## 1. Helper Functions

In [3]:
import numpy as np
import pandas as pd


def add_missingness_features(
    df,
    exclude_cols=None,
    min_missing_rate=0.01
):
    """
    Add missingness indicators only for columns with meaningful missingness.
    """
    df = df.copy()
    exclude_cols = set(exclude_cols or [])

    candidate_cols = [
        c for c in df.columns
        if c not in exclude_cols
        and df[c].isna().mean() >= min_missing_rate
    ]

    missing_features = pd.DataFrame(
        {
            f"{c}_missing": df[c].isna().astype("int8")
            for c in candidate_cols
        },
        index=df.index
    )

    return pd.concat([df, missing_features], axis=1)


def add_log_features(df, columns):
    """
    Add log1p features for non-negative numerical columns.
    """
    df = df.copy()

    for col in columns:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors="coerce")
            values = values.clip(lower=0)
            df[f"{col}_log1p"] = np.log1p(values)

    return df


def add_frequency_features(df, columns, suffix="_freq"):
    """
    Frequency encoding.

    For a train/test workflow, calculate the mapping on the training
    data only and apply it to validation/test data.
    """
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            continue

        frequencies = df[col].value_counts(dropna=False)
        df[f"{col}{suffix}"] = (
            df[col].map(frequencies).fillna(0).astype("float32")
        )

    return df

In [4]:
def fit_frequency_maps(train_df, columns):
    frequency_maps = {}

    for col in columns:
        if col in train_df.columns:
            frequency_maps[col] = train_df[col].value_counts(
                dropna=False
            ).to_dict()

    return frequency_maps


def apply_frequency_maps(df, frequency_maps, suffix="_freq"):
    df = df.copy()

    for col, mapping in frequency_maps.items():
        df[f"{col}{suffix}"] = (
            df[col].map(mapping).fillna(0).astype("float32")
        )

    return df

## 2. Sparkov Feature engineering:

From EDA, the strongest patterns are:
* transaction amount
* hour of day / late-night activity
* merchant category
* card history
* distance btwn cardholder and merchant location

In [5]:
def engineer_sparkov_base_features(df):
    df = df.copy()

    # ---------------------------------------------------------
    # Timestamp features
    # ---------------------------------------------------------
    df["trans_date_trans_time"] = pd.to_datetime(
        df["trans_date_trans_time"],
        errors="coerce"
    )

    timestamp = df["trans_date_trans_time"]

    df["trans_year"] = timestamp.dt.year.astype("int16")
    df["trans_month"] = timestamp.dt.month.astype("int8")
    df["trans_day"] = timestamp.dt.day.astype("int8")
    df["trans_dayofweek"] = timestamp.dt.dayofweek.astype("int8")
    df["trans_dayofyear"] = timestamp.dt.dayofyear.astype("int16")
    df["trans_hour"] = timestamp.dt.hour.astype("int8")
    df["trans_minute"] = timestamp.dt.minute.astype("int8")

    df["is_weekend"] = (
        timestamp.dt.dayofweek >= 5
    ).astype("int8")

    df["is_night"] = (
        (timestamp.dt.hour >= 22) |
        (timestamp.dt.hour <= 3)
    ).astype("int8")

    df["is_business_hour"] = (
        (timestamp.dt.hour >= 8) &
        (timestamp.dt.hour <= 18)
    ).astype("int8")

    # Cyclical time representations
    df["hour_sin"] = np.sin(
        2 * np.pi * df["trans_hour"] / 24
    )
    df["hour_cos"] = np.cos(
        2 * np.pi * df["trans_hour"] / 24
    )

    df["dow_sin"] = np.sin(
        2 * np.pi * df["trans_dayofweek"] / 7
    )
    df["dow_cos"] = np.cos(
        2 * np.pi * df["trans_dayofweek"] / 7
    )

    # ---------------------------------------------------------
    # Amount features
    # ---------------------------------------------------------
    df["amt_log1p"] = np.log1p(df["amt"].clip(lower=0))
    df["amt_is_round"] = (
        np.isclose(df["amt"] % 1, 0)
    ).astype("int8")

    df["amt_cents"] = (
        np.round((df["amt"] % 1) * 100)
        .astype("int16")
    )

    df["amt_bucket"] = pd.cut(
        df["amt"],
        bins=[-np.inf, 10, 25, 50, 100, 250, 500, 1000, np.inf],
        labels=False
    ).astype("Int64")

    # ---------------------------------------------------------
    # Age features
    # ---------------------------------------------------------
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

    df["customer_age"] = (
        timestamp.dt.year - df["dob"].dt.year
        - (
            (timestamp.dt.month < df["dob"].dt.month) |
            (
                (timestamp.dt.month == df["dob"].dt.month) &
                (timestamp.dt.day < df["dob"].dt.day)
            )
        ).astype("int8")
    )

    df["customer_age"] = df["customer_age"].clip(0, 120)

    # ---------------------------------------------------------
    # Geographic features
    # ---------------------------------------------------------
    df["lat_diff"] = df["lat"] - df["merch_lat"]
    df["long_diff"] = df["long"] - df["merch_long"]

    df["lat_diff_abs"] = df["lat_diff"].abs()
    df["long_diff_abs"] = df["long_diff"].abs()

    # Haversine distance in kilometres
    lat1 = np.radians(df["lat"])
    lon1 = np.radians(df["long"])
    lat2 = np.radians(df["merch_lat"])
    lon2 = np.radians(df["merch_long"])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    df["card_merchant_distance_km"] = (
        6371 * 2 * np.arcsin(np.sqrt(a))
    )

    df["distance_log1p"] = np.log1p(
        df["card_merchant_distance_km"].clip(lower=0)
    )

    # ---------------------------------------------------------
    # Population-related features
    # ---------------------------------------------------------
    df["city_pop_log1p"] = np.log1p(
        df["city_pop"].clip(lower=0)
    )

    df["amt_per_person"] = (
        df["amt"] /
        (df["city_pop"].replace(0, np.nan))
    )

    df["amt_per_person_log1p"] = np.log1p(
        df["amt_per_person"].clip(lower=0)
    )

    # ---------------------------------------------------------
    # Simple interactions
    # ---------------------------------------------------------
    df["amt_x_night"] = df["amt_log1p"] * df["is_night"]
    df["amt_x_weekend"] = df["amt_log1p"] * df["is_weekend"]
    df["distance_x_night"] = (
        df["distance_log1p"] * df["is_night"]
    )

    return df

In [6]:
sparkov_train_fe = engineer_sparkov_base_features(sparkov_train)
sparkov_test_fe = engineer_sparkov_base_features(sparkov_test)

#### Sparkov Frequency Features
* capture how common an entity appears

In [7]:
sparkov_frequency_cols = [
    "cc_num",
    "merchant",
    "category",
    "city",
    "state",
    "zip",
    "job",
    "street",
]

In [8]:
sparkov_freq_maps = fit_frequency_maps(
    sparkov_train_fe,
    sparkov_frequency_cols
)

sparkov_train_fe = apply_frequency_maps(
    sparkov_train_fe,
    sparkov_freq_maps
)

sparkov_test_fe = apply_frequency_maps(
    sparkov_test_fe,
    sparkov_freq_maps
)

#### Sparkov past-only card history features
* as `cc_num` has a long transaction history

In [9]:
def add_sparkov_past_history_features(df):
    df = df.copy()

    df = df.sort_values(
        ["cc_num", "trans_date_trans_time"]
    )

    group = df.groupby("cc_num", sort=False)

    # Number of earlier transactions by the card
    df["card_previous_txn_count"] = group.cumcount()

    # Time since previous transaction
    df["seconds_since_card_txn"] = (
        group["unix_time"].diff()
    )

    df["hours_since_card_txn"] = (
        df["seconds_since_card_txn"] / 3600
    )

    df["days_since_card_txn"] = (
        df["seconds_since_card_txn"] / 86400
    )

    # Previous amount
    df["previous_card_amt"] = group["amt"].shift(1)

    df["amount_change_from_previous"] = (
        df["amt"] - df["previous_card_amt"]
    )

    df["amount_ratio_to_previous"] = (
        df["amt"] /
        df["previous_card_amt"].replace(0, np.nan)
    )

    # Expanding historical amount statistics
    historical_amt = group["amt"].shift(1)

    df["card_previous_amt_mean"] = (
        historical_amt
        .groupby(df["cc_num"])
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
        .reindex(df.index)
    )

    df["card_previous_amt_std"] = (
        historical_amt
        .groupby(df["cc_num"])
        .expanding()
        .std()
        .reset_index(level=0, drop=True)
        .reindex(df.index)
    )

    # Restore original row order
    df = df.sort_index()

    return df

note: ensure that test history is only build from the end of training + earlier test transactions

In [10]:
sparkov_all = pd.concat(
    [
        sparkov_train.assign(_dataset="train"),
        sparkov_test.assign(_dataset="test"),
    ],
    axis=0,
    ignore_index=True
)

sparkov_all = engineer_sparkov_base_features(sparkov_all)
sparkov_all = add_sparkov_past_history_features(sparkov_all)

sparkov_train_fe = (
    sparkov_all[sparkov_all["_dataset"] == "train"]
    .drop(columns="_dataset")
)

sparkov_test_fe = (
    sparkov_all[sparkov_all["_dataset"] == "test"]
    .drop(columns="_dataset")
)

## 3. IEEE-CIS feature engineering

focus on:
* TransactionDT time features
* ProductCD, card, address and email relationships
* missingness patterns
* identity/device features
* carefully constructed card-history features
* D-column interpretation

#### Merge transaction and identity tables

In [11]:
def merge_ieee_tables(transaction_df, identity_df):
    transaction_df = transaction_df.copy()
    identity_df = identity_df.copy()

    identity_df = identity_df.drop_duplicates(
        subset=["TransactionID"]
    )

    df = transaction_df.merge(
        identity_df,
        on="TransactionID",
        how="left",
        validate="one_to_one",
        suffixes=("", "_identity")
    )

    df["has_identity"] = (
        df["TransactionID"].isin(
            set(identity_df["TransactionID"])
        )
    ).astype("int8")

    return df

In [12]:
ieee_train = merge_ieee_tables(
    ieee_transaction,
    ieee_identity
)

#### IEEE-CIS base features

In [13]:
def engineer_ieee_base_features(df):
    df = df.copy()

    # ---------------------------------------------------------
    # Relative time features
    # ---------------------------------------------------------
    df["relative_day"] = (
        df["TransactionDT"] // 86400
    ).astype("int16")

    df["seconds_in_day"] = (
        df["TransactionDT"] % 86400
    ).astype("int32")

    df["hour"] = (
        df["seconds_in_day"] // 3600
    ).astype("int8")

    df["minute"] = (
        (df["seconds_in_day"] % 3600) // 60
    ).astype("int8")

    df["is_night"] = (
        (df["hour"] >= 22) |
        (df["hour"] <= 5)
    ).astype("int8")

    df["is_business_hour"] = (
        (df["hour"] >= 8) &
        (df["hour"] <= 18)
    ).astype("int8")

    df["is_weekend_proxy"] = (
        (df["relative_day"] % 7 >= 5)
    ).astype("int8")

    # Cyclical time
    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )
    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["day_sin"] = np.sin(
        2 * np.pi * df["relative_day"] / 7
    )
    df["day_cos"] = np.cos(
        2 * np.pi * df["relative_day"] / 7
    )

    # ---------------------------------------------------------
    # Amount features
    # ---------------------------------------------------------
    df["TransactionAmt_log1p"] = np.log1p(
        df["TransactionAmt"].clip(lower=0)
    )

    df["TransactionAmt_is_round"] = (
        np.isclose(df["TransactionAmt"] % 1, 0)
    ).astype("int8")

    df["TransactionAmt_cents"] = (
        np.round(
            (df["TransactionAmt"] % 1) * 100
        ).astype("int16")
    )

    df["amount_bucket"] = pd.cut(
        df["TransactionAmt"],
        bins=[
            -np.inf, 10, 25, 50, 100,
            250, 500, 1000, 5000, np.inf
        ],
        labels=False
    ).astype("Int64")

    # ---------------------------------------------------------
    # Missingness summary features
    # ---------------------------------------------------------
    feature_cols = [
        c for c in df.columns
        if c not in ["isFraud", "TransactionID"]
    ]

    df["missing_count"] = (
        df[feature_cols].isna().sum(axis=1)
    ).astype("int16")

    df["missing_fraction"] = (
        df[feature_cols].isna().mean(axis=1)
    ).astype("float32")

    # Missingness by feature families
    for prefix in ["D", "V", "C", "M", "id_"]:
        cols = [
            c for c in df.columns
            if c.startswith(prefix)
        ]

        if cols:
            df[f"{prefix}missing_count"] = (
                df[cols].isna().sum(axis=1)
            ).astype("int16")

            df[f"{prefix}missing_fraction"] = (
                df[cols].isna().mean(axis=1)
            ).astype("float32")

    # ---------------------------------------------------------
    # Email features
    # ---------------------------------------------------------
    email_cols = [
        c for c in ["P_emaildomain", "R_emaildomain"]
        if c in df.columns
    ]

    if email_cols:
        for col in email_cols:
            df[f"{col}_is_missing"] = (
                df[col].isna()
            ).astype("int8")

            df[f"{col}_provider"] = (
                df[col]
                .astype("string")
                .str.lower()
                .str.split(".", n=1)
                .str[0]
            )

        if (
            "P_emaildomain" in df.columns
            and "R_emaildomain" in df.columns
        ):
            df["email_domain_match"] = (
                df["P_emaildomain"]
                == df["R_emaildomain"]
            ).astype("int8")

    # ---------------------------------------------------------
    # Card and address relationships
    # ---------------------------------------------------------
    card_cols = [
        c for c in [
            "card1", "card2", "card3",
            "card4", "card5", "card6"
        ]
        if c in df.columns
    ]

    if card_cols:
        df["card_missing_count"] = (
            df[card_cols].isna().sum(axis=1)
        ).astype("int8")

    if "addr1" in df.columns and "addr2" in df.columns:
        df["address_missing_count"] = (
            df[["addr1", "addr2"]]
            .isna()
            .sum(axis=1)
            .astype("int8")
        )

        df["address_pair"] = (
            df["addr1"].astype("string")
            + "_"
            + df["addr2"].astype("string")
        )

    if "card4" in df.columns and "card6" in df.columns:
        df["card_type_pair"] = (
            df["card4"].astype("string")
            + "_"
            + df["card6"].astype("string")
        )

    # ---------------------------------------------------------
    # Identity/device features
    # ---------------------------------------------------------
    if "DeviceInfo" in df.columns:
        device_text = (
            df["DeviceInfo"]
            .astype("string")
            .str.lower()
        )

        df["deviceinfo_length"] = (
            device_text.str.len()
        )

        df["deviceinfo_token_count"] = (
            device_text.str.split().str.len()
        )

        df["deviceinfo_has_android"] = (
            device_text.str.contains(
                "android",
                na=False
            )
        ).astype("int8")

        df["deviceinfo_has_iphone"] = (
            device_text.str.contains(
                "iphone|ios",
                na=False
            )
        ).astype("int8")

    if "id_30" in df.columns:
        df["id30_os_family"] = (
            df["id_30"]
            .astype("string")
            .str.lower()
            .str.extract(
                r"(windows|ios|android|mac|linux)",
                expand=False
            )
        )

    if "id_31" in df.columns:
        df["id31_browser_family"] = (
            df["id_31"]
            .astype("string")
            .str.lower()
            .str.extract(
                r"(chrome|safari|firefox|edge|opera|ie)",
                expand=False
            )
        )

    if "id_30" in df.columns and "id_31" in df.columns:
        df["os_browser_pair"] = (
            df["id_30"].astype("string")
            + "_"
            + df["id_31"].astype("string")
        )

    # ---------------------------------------------------------
    # D-column derived features
    # ---------------------------------------------------------
    d_cols = [
        c for c in df.columns
        if c.startswith("D")
        and c[1:].isdigit()
    ]

    if d_cols:
        df["D_nonmissing_count"] = (
            df[d_cols].notna().sum(axis=1)
        ).astype("int16")

        df["D_zero_count"] = (
            (df[d_cols] == 0).sum(axis=1)
        ).astype("int16")

    # Features identified by the EDA as more useful
    for col in ["D1", "D3", "D4", "D10", "D15"]:
        if col in df.columns:
            df[f"{col}_missing"] = (
                df[col].isna()
            ).astype("int8")

    if "D1" in df.columns:
        df["card_first_use_day_proxy"] = (
            df["relative_day"] - df["D1"]
        )

    if "D3" in df.columns:
        df["D3_log1p"] = np.log1p(
            df["D3"].clip(lower=0)
        )

    return df

In [14]:
ieee_train_fe = engineer_ieee_base_features(ieee_train)

#### IEEE-CIS frequency features

In [15]:
ieee_frequency_cols = [
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "ProductCD",
    "DeviceInfo",
    "id_30",
    "id_31",
    "address_pair",
    "card_type_pair",
]

In [16]:
ieee_freq_maps = fit_frequency_maps(
    ieee_train_fe,
    ieee_frequency_cols
)

ieee_train_fe = apply_frequency_maps(
    ieee_train_fe,
    ieee_freq_maps
)

#### IEEE-CIS card proxy key
* from exploratory analysis, `card1 + addr1 + D1n` can help to differentiate cards

In [ ]:
def add_ieee_card_proxy_features(df):
    df = df.copy()

    if "D1" in df.columns:
        df["D1n"] = (
            df["relative_day"] - df["D1"]
        )

        df["card_proxy_key"] = (
            df["card1"].astype("string")
            + "_"
            + df["addr1"].astype("string")
            + "_"
            + df["D1n"].round().astype("string")
        )

        df["card_proxy_key_missing"] = (
            df["card_proxy_key"]
            .isin(["<NA>_<NA>_<NA>"])
        ).astype("int8")

    return df

In [ ]:
ieee_train_fe = add_ieee_card_proxy_features(
    ieee_train_fe
)

In [ ]:
card_proxy_frequency_map = fit_frequency_maps(
    ieee_train_fe,
    ["card_proxy_key"]
)

ieee_train_fe = apply_frequency_maps(
    ieee_train_fe,
    card_proxy_frequency_map,
    suffix="_frequency"
)

#### Past-only IEEE-CIS history features
* from EDA, it is noted that D3 refers to the number of days since the last transaction
* D3 is said to match the previoius transaction gap for 94.6% of checkable transactions

Thus,
* this code seeks to confirm if a reconstructed card is sensible

In [ ]:
def add_ieee_past_history_features(df):
    df = df.copy()

    sort_cols = ["TransactionDT", "TransactionID"]
    df = df.sort_values(sort_cols)

    history_keys = [
        "card_proxy_key",
        "card1",
        "addr1",
        "P_emaildomain",
        "R_emaildomain",
        "DeviceInfo",
    ]

    history_keys = [
        c for c in history_keys
        if c in df.columns
    ]

    for key in history_keys:
        group = df.groupby(key, sort=False)

        df[f"{key}_previous_count"] = (
            group.cumcount()
        )

        df[f"{key}_previous_time"] = (
            group["TransactionDT"].shift(1)
        )

        df[f"{key}_seconds_since_previous"] = (
            df["TransactionDT"]
            - df[f"{key}_previous_time"]
        )

        df[f"{key}_days_since_previous"] = (
            df[f"{key}_seconds_since_previous"] / 86400
        )

    # Compare D3 with the observed previous transaction gap
    if (
        "card_proxy_key" in df.columns
        and "D3" in df.columns
    ):
        previous_time = (
            df.groupby("card_proxy_key")["TransactionDT"]
            .shift(1)
        )

        observed_gap_days = (
            df["TransactionDT"] - previous_time
        ) / 86400

        df["D3_gap_difference"] = (
            df["D3"] - observed_gap_days
        )

        df["D3_matches_previous_gap"] = (
            np.isclose(
                df["D3"],
                observed_gap_days,
                equal_nan=False,
                atol=1.0
            )
        ).astype("int8")

    return df.sort_index()

In [ ]:
ieee_train_fe = add_ieee_past_history_features(
    ieee_train_fe
)